# 多模态检索样例

用户提供测试数据，csv 格式(id, url, create_time, ...)

以图搜图, 用户选择本地图片,库中检索最相似图片集

以文搜图，返回的 topK，每条记录就包含上述信息, 可以带部分标签过滤

<p align="center">
  <img src="data/架构图.png" width="60%"/>
</p>


## 图片入库

### 配置 Lindorm 实例
阿里云控制台创建 lindorm 实例，开通搜索、向量、AI引擎
config.py 中配置 lindorm 实例地址、用户名和密码等信息

### 配置模型
* 多模态 embedding 模型: qwen3-vl-embedding
* vl 模型: qwen3-vl-plus
* rerank 模型: qwen3-rerank

In [ ]:
import ipywidgets as widgets
import logging
from IPython.display import display, clear_output
from src.lindorm import Lindorm
from src.prompt import VL_PROMPT, REWRITE_SUMMARY_PROMPT

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('lindorm.log', mode='a', encoding='utf-8'),
        logging.StreamHandler()  # 这个会输出到终端
    ]
)

# 禁用 OpenSearch / Elasticsearch 的 INFO 和 WARNING 日志
logging.getLogger("opensearch").setLevel(logging.ERROR)
logging.getLogger("elasticsearch").setLevel(logging.ERROR)

# 可选：也可以关闭 urllib3 的连接池日志（常见噪音来源）
logging.getLogger("urllib3.connectionpool").setLevel(logging.ERROR)

vl_model = widgets.Dropdown(
    options=['qwen3-vl-plus', 'qwen3-vl-flash'],
    value='qwen3-vl-plus',
    description='视觉理解模型:',
)
vl_model.style.description_width = 'initial'

embedding_model = widgets.Dropdown(
    options=['qwen2.5-vl-embedding', 'qwen3-vl-embedding'],
    value='qwen2.5-vl-embedding',
    description='多模态embedding模型:',
)
embedding_model.style.description_width = 'initial'

rewrite_model = widgets.Dropdown(
    options=['qwen-plus', 'qwen-max', 'qwen3-max'],
    value='qwen-plus',
    description='文本生成模型:',
)
rewrite_model.style.description_width = 'initial'

rerank_model = widgets.Dropdown(
    options=['qwen3-rerank', 'gte-rerank-v2'],
    value='qwen3-rerank',
    description='多模态rerank模型:',
)
rerank_model.style.description_width = 'initial'

index_input = widgets.Text(
    value='multimodal_retrieval_index',
    placeholder='index_name',
    description='index name:',
    disabled=False
)
index_input.style.description_width = 'initial'

# 创建下拉框
clear_dropdown = widgets.Dropdown(
    options=[('True', True), ('False', False)],
    value=False,  # 默认值为 False
    description='强制清库:',
)

# 显示控件
display(vl_model, rewrite_model, embedding_model, rerank_model, index_input, clear_dropdown)

### 配置索引

* 新建 lindorm 索引，配置 schema 信息

In [ ]:
import json
from index import get_index_body

# 触发按钮, 自动触发创建 lindorm 索引

# 避免重复 display 导致的前端混乱
output_area = widgets.Output()

def create_index(change):
    with output_area:
        output_area.clear_output()  # 清空上次输出
        try:
            lindorm = Lindorm(index_input.value)
            res = lindorm.lindormSearch.get_index()
            if res:
                logging.info(f"clear_old {clear_dropdown.value}, index exists: {json.dumps(res, indent=2, ensure_ascii=False)}")
                if not clear_dropdown.value:
                    return
                logging.info(f"index deleted: {lindorm.lindormSearch.drop_index()}")

            index_body = get_index_body(1024)
            logging.info(f"create index: {lindorm.lindormSearch.create_search_index(index_body)}")
        except Exception as e:
            logging.error("Create index failed", exc_info=True)


index_button = widgets.Button()
index_button.description = '创建索引表'
index_button.on_click(create_index)


display(index_button, output_area)


### 信息扫描入库

* 配置 csv 文件

* 图片智能处理，入 lindorm 库
    * qwen-vl 图片识别，转成json 文本
    * url 进行图片 embedding, 采用多模态 embedding 模型
    * 描述文本、embedding 向量值与 csv 中基础信息共同入库

In [ ]:
import io
import re
import csv
import time
import threading
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

def parse_markdown_json(s):
    """
    从形如 ```json{...}``` 的字符串中提取并解析 JSON
    """
    if not isinstance(s, str):
        raise TypeError("Input must be a string")

    # 移除首尾空白后匹配
    s = s.strip()
    pattern = r'^```(?:json)?\s*\n?(.*?)\n?\s*```$'
    match = re.search(pattern, s, re.DOTALL | re.IGNORECASE)

    if not match:
        raise ValueError("Cannot find valid JSON in code block")

    try:
        return json.loads(match.group(1))
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON content: {e}")

def safe_json_loads(s):
    try:
        return json.loads(s)
    except json.JSONDecodeError as e:
        print(f"JSON 解析失败: {e}")
        return None

def retry_vl_picture_withdraw(url, lindorm: Lindorm, retry_times=5):
    for i in range(retry_times):
        try:
            return lindorm.lindormAI.vl_picture_withdraw(url, vl_model.value, VL_PROMPT)
        except Exception as e:
            logging.error(f"vl_picture_withdraw failed, retry {i+1}/{retry_times}", exc_info=True)
            time.sleep(3)
    raise Exception("vl_picture_withdraw failed")


def process_row_safe(row, lindorm: Lindorm):
    """线程安全的行处理函数（遇到异常立即抛出）"""
    doc_id = row.get('id')
    if not doc_id:
        raise ValueError(f"Missing 'id' in row: {row}")

    row_copy = dict(row)
    del row_copy['id']

    res = lindorm.lindormSearch.get_doc(doc_id)
    if res:
        logging.debug(f"doc exists, skip row: {doc_id}")
        return f"Skip: {doc_id}"

    try:
        embedding = lindorm.lindormAI.embedding("image", row_copy['url'], embedding_model.value)
        vl_desc = retry_vl_picture_withdraw(row_copy['url'], lindorm)
        vl_desc_json = safe_json_loads(vl_desc)
        row_copy['embedding'] = embedding
        # 改写优化 vl 返回的描述
        vl_desc_rewrite = lindorm.lindormAI.rewrite_text(vl_desc_json.get('content'), rewrite_model.value, REWRITE_SUMMARY_PROMPT)
        row_copy['img_desc'] = vl_desc_rewrite
        lindorm.lindormSearch.write_doc(row_copy, doc_id)
        return f"Success: {doc_id}"
    except Exception as e:
        logging.error(f"Critical error processing row {row}: {e}", exc_info=True)
        # 直接抛出异常以终止整个流程
        raise


# CSV 文件处理函数
def process_csv_file(change):
    """逐行读取并处理CSV文件"""
    with csv_output:
        csv_output.clear_output()
        if not change['new']:
            print('请选择一个CSV文件')
            return
        lindorm = Lindorm(index_input.value)

        # 获取上传的文件内容
        uploaded_file = change['new'][0]
        file_content = uploaded_file['content']
        file_name = uploaded_file['name']

        print(f'开始处理文件: {file_name}')

        # 将字节内容转换为文本
        text_content = bytes(file_content).decode('utf-8')
        
        # 计算总行数（减1是为了排除表头）
        csv_count = sum(1 for _ in csv.reader(io.StringIO(text_content))) - 1
        print(f'CSV文件共有 {csv_count} 行数据')
        
        # 重新创建reader用于实际处理
        csv_reader = csv.DictReader(io.StringIO(text_content))

        # 使用线程池并发处理
        max_workers = 5
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            # 提交所有任务
            future_to_row = {
                executor.submit(process_row_safe, row, lindorm): row
                for row in csv_reader
            }

            # 线程安全地更新 tqdm 进度
            pbar = tqdm(total=csv_count, desc='处理CSV (并发)')
            try:
                for future in as_completed(future_to_row):
                    future.result()  # 此处会抛出异常
                    pbar.update(1)
            except Exception as e:
                # 取消所有未完成的任务
                for future in future_to_row:
                    future.cancel()
                pbar.close()
                raise  # 重新抛出异常终止流程
            else:
                pbar.close()

        print(f'\n文件处理完成! 共处理 {csv_count} 行数据')

# 创建文件选择框, 选择 csv 文件
csv_file_uploader = widgets.FileUpload(
    accept='.csv',  # 只接受 CSV 文件
    multiple=False,  # 只允许上传单个文件
    description='选择CSV文件:',
    disabled=False
)
csv_file_uploader.style.description_width = 'initial'

# 创建输出区域显示处理进度
csv_output = widgets.Output()

# 绑定文件上传事件
csv_file_uploader.observe(process_csv_file, names='value')

display(csv_file_uploader, csv_output)

### 构建磁盘索引
* 创建磁盘索引
* 循环等待索引创建完成
* 一次性操作，后续新增数据会自动新增索引条目，实时可见

In [ ]:
import json

# 避免重复 display 导致的前端混乱
output_area = widgets.Output()

def build_ivfpq_index(change):
    with output_area:
        output_area.clear_output()  # 清空上次输出
        try:
            lindorm = Lindorm(index_input.value)
            res = lindorm.lindormSearch.query_index_build_states("embedding")
            if not res['payload']:
                logging.info(f"start build index: {lindorm.lindormSearch.build_index('embedding')}")
            while True:
                res = lindorm.lindormSearch.query_index_build_states("embedding")
                logging.info(f"build index status: {json.dumps(res, indent=2, ensure_ascii=False)}")
                if "FINISH" in res['payload'][0]:
                    logging.info(f"build index success")
                    break
                time.sleep(5)
        except Exception as e:
            logging.error("Build index failed", exc_info=True)

build_index_button = widgets.Button()
build_index_button.description = '构建磁盘索引'
build_index_button.on_click(build_ivfpq_index)

display(build_index_button, output_area)


## 多模态检索

以文搜图

以图搜图

In [ ]:
import base64
import uuid
import json
import io
import os
import logging
import ipywidgets as widgets
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, clear_output, HTML
from src.lindorm import Lindorm

lindorm = Lindorm(index_input.value)

vector_proportion = widgets.FloatText(
    value=0.3,
    placeholder=0.3,
    min=0.0,     # 最小值
    max=1.0,     # 最大值
    step=0.1,    # 步长
    description='融合检索向量占比:',
    disabled=False
)
vector_proportion.style.description_width = 'initial'

min_score = widgets.FloatText(
    value=0.50,
    placeholder=0.50,
    min=0.0,     # 最小值
    max=1.0,     # 最大值
    step=0.02,    # 步长
    description='最小相似度阈值:',
    disabled=False
)
min_score.style.description_width = 'initial'

decline_rate = widgets.FloatText(
    value=0.3,
    placeholder=0.3,
    min=0.0,     # 最小值
    max=1.0,     # 最大值
    step=0.02,    # 步长
    description='相似度下降幅度阈值:',
    disabled=False
)
decline_rate.style.description_width = 'initial'

top_k = widgets.IntText(
    value=50,
    placeholder=50,
    min=1,        # 最小值
    max=50,     # 最大值
    step=10,    # 步长
    description='topK:',
    disabled=False
)
top_k.style.description_width = 'initial'

recall_count = widgets.IntText(
    value=3,
    placeholder=3,
    min=1,        # 最小值
    max=50,     # 最大值
    step=1,    # 步长
    description='topK:',
    disabled=False
)
recall_count.style.description_width = 'initial'

# 创建输入框和按钮
text_input = widgets.Text(
    value='',
    placeholder='输入关键字',
    description='以文搜图:',
    disabled=False
)

file_input = widgets.FileUpload(
    accept='image/*',
    description='以图搜图',
    multiple=False  # 只允许选择一张图片
)

# 是否需要查询改写
need_rewrite = widgets.Dropdown(
    options=[('否', False), ('qwen3-plus', '')],
    value=False,  # 默认值为 False
    description='rerank优化:',
)

# 定义搜索按钮
search_buttons = [
    # widgets.Button(description="纯向量检索", button_style=''),
    widgets.Button(description="RRF融合检索", button_style=''),
    # widgets.Button(description="全文检索", button_style='')
]

output = widgets.Output()

def rerank_hits(hits, text, rerank_model):
    description_chunks = []
    for hit in hits:
        description_chunks.append(hit.get('_source').get("img_desc"))
    new_hits = []
    result = lindorm.lindormAI.rerank_text(text, description_chunks, rerank_model, recall_count.value)
    for rerank_hit in result:
        hit = hits[rerank_hit.get('index')]
        hit['_score'] = rerank_hit.get('relevance_score')
        new_hits.append(hit)
    return new_hits
    
def cut_hits(hits, count: int):
    new_hits = []
    last_score = 0
    for hit in hits:
        cur_score = hit.get('_score')
        if last_score == 0:
            last_score = cur_score
        rate = (last_score - cur_score) / last_score
        # print(f"cur_score {cur_score}, last_score {last_score}, rate {rate}")
        if rate >= decline_rate.value:
            break
        new_hits.append(hit)
        last_score = cur_score
        if len(new_hits) >= count:
            break
    return new_hits

def show_pictures(hits):
    # 生成唯一标识符前缀
    unique_prefix = str(uuid.uuid4())[:8]

    html_parts = [
        '<link href="https://cdnjs.cloudflare.com/ajax/libs/tailwindcss/2.2.19/tailwind.min.css" rel="stylesheet">',
        '<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.0.0-beta3/css/all.min.css">',
        '<style type="text/tailwindcss">',
        '@layer utilities {',
        '.content-auto { content-visibility: auto; }',
        '.zoom-transition { transition: transform 0.3s ease; }',
        '}',
        '</style>',
        '<div style="display: flex; flex-wrap: wrap; gap: 15px;">'
    ]

    for i, hit in enumerate(hits):
        # 从 _source 中获取 url 而不是从 _id
        id = hit.get('_id')
        url = hit.get('_source', {}).get('url', '')
        score = hit.get('_score')
        description_text = hit.get('_source', {}).get('img_desc', '暂无描述')
        create_time = hit.get('_source', {}).get('create_time', '未知时间')

        # 使用唯一前缀和索引组合生成ID
        desc_id = f"{unique_prefix}-desc-{i}"
        btn_id = f"{unique_prefix}-btn-{i}"
        img_id = f"{unique_prefix}-img-{i}"

        html_parts.append(f'''
        <div style="flex: 0 0 calc(33.33% - 15px); box-sizing: border-box; margin-bottom: 15px;">
            <div style="border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                <img id="{img_id}" src="{url}" style="width: 100%; height: 200px; object-fit: cover; cursor: zoom-in;"
                     alt="Image" onclick="toggleImageZoom('{img_id}', '{unique_prefix}')"
                     ondblclick="event.stopPropagation()">
                <div style="padding: 10px; text-align: center; font-size: 14px; background-color: #f9f9f9;">
                    {id}  {score:.5f}
                </div>
                <div class="p-4">
                    <div id="{desc_id}" class="max-h-0 overflow-hidden opacity-0 transition-all duration-500 ease-in-out">
                        <p class="text-gray-700 text-sm leading-relaxed py-2">创建时间: {create_time}</p>
                        <p class="text-gray-700 text-sm leading-relaxed py-2">{description_text}</p>
                    </div>

                    <button id="{btn_id}" onclick="toggleDescription('{desc_id}', '{btn_id}')"
                            class="mt-2 text-blue-600 hover:text-blue-800 text-sm font-medium flex items-center w-full justify-center py-2 px-3 border border-blue-600 rounded hover:bg-blue-50 transition-colors">
                        <span class="button-text">查看详情</span>
                        <i class="fas fa-chevron-down ml-2 transition-transform duration-300"></i>
                    </button>
                </div>
            </div>
        </div>
        ''')

    # 添加遮罩层和JavaScript函数
    html_parts.append(f"""
    </div>

    <!-- 图片放大遮罩层 -->
    <div id="{unique_prefix}-zoom-overlay" class="fixed inset-0 bg-black bg-opacity-80 z-50 hidden items-center justify-center">
        <img id="{unique_prefix}-zoomed-img" src="" class="max-w-[90%] max-h-[90vh] object-contain cursor-zoom-out"
             onclick="closeZoom('{unique_prefix}')">
    </div>

    <script>
        // 描述展开/收起
        function toggleDescription(descId, btnId) {{
            const desc = document.getElementById(descId);
            const btn = document.getElementById(btnId);
            const icon = btn.querySelector('i');
            const btnText = btn.querySelector('.button-text');

            if (desc.classList.contains('max-h-0')) {{
                desc.classList.remove('max-h-0', 'opacity-0');
                desc.classList.add('max-h-96', 'opacity-100');
                if(btnText) btnText.textContent = '收起详情';
                icon.classList.add('rotate-180');
            }} else {{
                desc.classList.remove('max-h-96', 'opacity-100');
                desc.classList.add('max-h-0', 'opacity-0');
                if(btnText) btnText.textContent = '查看详情';
                icon.classList.remove('rotate-180');
            }}
        }}

        // 图片放大
        function toggleImageZoom(imgId, prefix) {{
            const img = document.getElementById(imgId);
            const overlay = document.getElementById(`${{prefix}}-zoom-overlay`);
            const zoomedImg = document.getElementById(`${{prefix}}-zoomed-img`);

            // 设置放大图片的源
            zoomedImg.src = img.src;

            // 显示遮罩层并添加动画
            overlay.classList.remove('hidden');
            overlay.classList.add('flex');

            // 添加ESC键监听
            document.addEventListener('keydown', handleKeydown);

            function handleKeydown(e) {{
                if (e.key === 'Escape') {{
                    closeZoom(prefix);
                    document.removeEventListener('keydown', handleKeydown);
                }}
            }}
        }}

        // 关闭放大
        function closeZoom(prefix) {{
            const overlay = document.getElementById(`${{prefix}}-zoom-overlay`);
            overlay.classList.remove('flex');
            overlay.classList.add('hidden');
            document.removeEventListener('keydown', handleKeydown);
        }}
    </script>
    """)

    display(HTML(''.join(html_parts)))

        
def show_hits(hits, keyword):
    if hits is None or len(hits) == 0:
        print("图片搜索失败")
        return
    
    hits = rerank_hits(hits, keyword, rerank_model.value)
    print("✅图片搜索成功，已进行rerank")

    new_hits = cut_hits(hits, recall_count.value)
    count = len(new_hits)
    
    if count == 0:
        print("图片搜索失败")
        return
    show_pictures(new_hits)
              
# 定义按钮点击事件处理函数
def on_button_clicked(b):
    with output:
        clear_output()  # 清除上次输出
        if not text_input.value:
            print("请先输入描述文字")
            return
        text_embedding = lindorm.lindormAI.embedding("text", text_input.value, embedding_model.value)
        if b.description == "纯向量检索":
            hits = lindorm.lindormSearch.knn_search(text_embedding, "embedding", True, min_score.value, top_k.value)
        elif b.description == "RRF融合检索":
            hits = lindorm.lindormSearch.rrf_search(text_input.value, "img_desc", text_embedding, "embedding", True, vector_proportion.value, min_score.value, top_k.value)
        elif b.description == "全文检索":
            hits = lindorm.lindormSearch.full_text_search(text_input.value, "img_desc", True, top_k.value)
        else:
            logging.error("无效的按钮描述")
        show_hits(hits, text_input.value)
        
# 定义图片显示函数
def show_uploaded_image(change):
    with output:
        clear_output()  # 清除上次输出
        if change['new']:
            # 获取上传的文件内容
            uploaded_file = change['new'][0]
            content = uploaded_file['content']
            # 使用 PIL 打开图片
            image = Image.open(io.BytesIO(content))
            # 使用 matplotlib 显示图片
            plt.imshow(image)
            plt.axis('off')  # 不显示坐标轴
            plt.show()
            base64_image = base64.b64encode(content).decode('utf-8')
            image_format = image.format
            image_data = f"data:image/{image_format};base64,{base64_image}"
            image_embedding = lindorm.lindormAI.embedding("image", image_data, embedding_model.value)
            hits = lindorm.lindormSearch.knn_search(image_embedding, "embedding", True, min_score.value, recall_count.value)
            show_pictures(hits)
            
        
for button in search_buttons:
    button.on_click(lambda b: on_button_clicked(b))
    
# 绑定文件上传事件
file_input.observe(show_uploaded_image, names='value')

display(vector_proportion, min_score, decline_rate, top_k, recall_count, text_input, *search_buttons, file_input, output)
